# 01 · Ingesta del DENUE 2020 y 2025

**Objetivo:** limpiar las dos ediciones del DENUE para los 4 municipios de la Zona Metropolitana de Querétaro y separar tres cosas:

| Archivo | Qué contiene | Para qué |
|---|---|---|
| `denue_zmq_AAAA-MM.parquet` | Todos los establecimientos, sin administración pública | Buscar sucesores (03) y generadores de tráfico (04) |
| `gobierno_zmq_AAAA-MM.parquet` | Solo administración pública (SCIAN 93) | Oficinas de gobierno como generador de tráfico (04) |
| `restaurantes_zmq.parquet` | Restaurantes (SCIAN 722) de las dos ediciones, en 10 tipos | Lo que se analiza |

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import h3
import mapas

ROOT = Path.cwd().parent
RAW, PROC, FIG = ROOT / "data/raw", ROOT / "data/processed", ROOT / "figures"
miles = FuncFormatter(lambda v, _: f"{v:,.0f}")

## 1. Configuración

- `BASE` y `FIN`: las dos ediciones con Censo Económico.
- `TIPOS_RESTAURANTE`: los 10 tipos que se analizan. Los centros nocturnos se juntan con los bares por su bajo número.

In [ ]:
BASE, FIN = "2020-04", "2025-05"
MUNICIPIOS_ZMQ = {6: "Corregidora", 8: "Huimilpan", 11: "El Marqués", 14: "Querétaro"}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 20.2, 21.0, -100.7, -99.9     # caja de la ZM (descarta errores de coordenadas)
RES = 8                                                              # hexágonos de ~0.74 km²

TIPOS_RESTAURANTE = {
    "722511": "A la carta / comida corrida", "722512": "Mariscos", "722513": "Antojitos",
    "722514": "Tacos y tortas", "722515": "Cafeterías y neverías", "722516": "Autoservicio",
    "722517": "Pizzas, hamburguesas y pollos", "722518": "Otros para llevar",
    "722519": "Otros de consumo inmediato", "722411": "Bares y centros nocturnos",
    "722412": "Bares y centros nocturnos",
}
COLS = ["id", "nom_estab", "raz_social", "codigo_act", "nombre_act", "per_ocu", "cve_ent", "cve_mun",
        "municipio", "ageb", "manzana", "nom_vial", "numero_ext", "numero_int", "latitud", "longitud", "fecha_alta"]

## 2. Limpiar cada edición

1. Se queda con los 4 municipios.
2. Separa la administración pública (SCIAN 93).
3. Descarta coordenadas fuera de la zona metropolitana y las anota en una bitácora.

In [ ]:
def preparar(etiqueta):
    df = pd.read_parquet(RAW / f"denue_qro_{etiqueta}.parquet")
    df.columns = df.columns.str.strip().str.lower()
    df = df[[c for c in COLS if c in df.columns]].copy()
    df["cve_ent"] = pd.to_numeric(df["cve_ent"], errors="coerce")
    df["cve_mun"] = pd.to_numeric(df["cve_mun"], errors="coerce")
    df = df[(df["cve_ent"] == 22) & df["cve_mun"].isin(MUNICIPIOS_ZMQ)]
    df["codigo_act"] = df["codigo_act"].astype(str).str.strip().str.zfill(6)
    df["latitud"] = pd.to_numeric(df["latitud"], errors="coerce")
    df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")

    dentro = df["latitud"].between(LAT_MIN, LAT_MAX) & df["longitud"].between(LON_MIN, LON_MAX)
    fuera = df.loc[~dentro, ["id", "nom_estab", "municipio", "latitud", "longitud"]].assign(edicion=etiqueta)
    df = df[dentro].assign(edicion=etiqueta, mun_nombre=lambda d: d["cve_mun"].map(MUNICIPIOS_ZMQ))

    gobierno = df[df["codigo_act"].str.startswith("93")]
    df = df[~df["codigo_act"].str.startswith("93")]
    df.to_parquet(PROC / f"denue_zmq_{etiqueta}.parquet", index=False)
    gobierno.to_parquet(PROC / f"gobierno_zmq_{etiqueta}.parquet", index=False)
    print(f"{etiqueta}: {len(df):,} establecimientos · {len(gobierno):,} de gobierno · {len(fuera)} fuera de la ZM")
    return df, fuera

limpios, fueras = zip(*(preparar(e) for e in (BASE, FIN)))
denue = pd.concat(limpios, ignore_index=True)
pd.concat(fueras, ignore_index=True).to_csv(PROC / "registros_fuera_de_zm.csv", index=False)

## 3. Restaurantes por tipo

El resultado se guarda en `restaurantes_zmq.parquet`. Los comedores no abiertos al público (escolares, comunitarios, de empresa) se excluyen en el notebook 03, al seguir a cada restaurante.

In [ ]:
rest = denue[denue["codigo_act"].isin(TIPOS_RESTAURANTE)].copy()
rest["tipo"] = rest["codigo_act"].map(TIPOS_RESTAURANTE)
rest.to_parquet(PROC / "restaurantes_zmq.parquet", index=False)

por_tipo = rest.groupby(["tipo", "edicion"]).size().unstack(fill_value=0)[[BASE, FIN]]
por_tipo["cambio %"] = ((por_tipo[FIN] / por_tipo[BASE] - 1) * 100).round(1)
por_tipo = por_tipo.sort_values(BASE, ascending=False)
b, f = por_tipo[BASE].sum(), por_tipo[FIN].sum()
print(f"Restaurantes {BASE}: {b:,} · {FIN}: {f:,} · cambio neto: {f - b:+,} ({f / b - 1:+.1%})")
por_tipo

## 4. Cambio por municipio

In [ ]:
por_mun = rest.groupby(["mun_nombre", "edicion"]).size().unstack(fill_value=0)[[BASE, FIN]]
por_mun["cambio %"] = ((por_mun[FIN] / por_mun[BASE] - 1) * 100).round(1)
por_mun.sort_values(BASE, ascending=False)

## 5. Gráfica · Restaurantes por tipo, 2020 contra 2025

In [ ]:
comp = por_tipo[[BASE, FIN]].iloc[::-1]
cambio = comp[FIN] / comp[BASE] - 1
fig, ax = plt.subplots(figsize=(11, 7))
y, h = np.arange(len(comp)), 0.38
ax.barh(y + h / 2, comp[BASE], h, color="#C9C2B4", label=BASE)
ax.barh(y - h / 2, comp[FIN], h, color="#2E6F95", label=FIN)
for yi, v, c in zip(y, comp[FIN], cambio):
    ax.text(v + comp.values.max() * 0.01, yi - h / 2, f"{v:,}  ({c:+.0%})", va="center", fontsize=10)
ax.set_yticks(y, comp.index); ax.xaxis.set_major_formatter(miles)
ax.set_xlim(0, comp.values.max() * 1.25); ax.set_xlabel("Restaurantes")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="lower right")
ax.set_title(f"Restaurantes por tipo · {BASE} vs {FIN}", loc="left")
plt.tight_layout(); plt.savefig(FIG / "01_restaurantes_por_tipo.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Mapa · Densidad de restaurantes en 2025

In [ ]:
fin = rest[rest["edicion"] == FIN]
conteo = pd.Series([h3.latlng_to_cell(la, lo, RES) for la, lo in zip(fin["latitud"], fin["longitud"])]).value_counts()
BORDES = [1, 2, 5, 10, 25, 50, conteo.max() + 1]
ETIQ = ["1", "2–4", "5–9", "10–24", "25–49", f"50–{conteo.max():,}"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#FFF3D6", "#FDD49E", "#FDAE6B", "#F16913", "#C2410C", "#7F2704"])
fig, ax, polys = mapas.base_mapa(conteo.index)
coll = mapas.capa_hex(ax, polys, array=conteo.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Restaurantes por hexágono (~0.74 km²)")
ax.set_title(f"Densidad de restaurantes · {FIN}", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "01_mapa_restaurantes.png", dpi=200, bbox_inches="tight"); plt.show()